In [2]:
trips_path = (
    "abfss://Fleet_Logistics_Engineering@onelake.dfs.fabric.microsoft.com/"
    "Fleet_Logistics_Lakehouse.Lakehouse/Files/Landing/trips.csv"
)

df_trips_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(trips_path)
)

StatementMeta(, 83a6f1df-6aca-48d1-879a-d5be832b38d4, 4, Finished, Available, Finished, False)

In [3]:
display(df_trips_raw)

df_trips_raw.printSchema()

print(f"Source records: {df_trips_raw.count()}")

StatementMeta(, 83a6f1df-6aca-48d1-879a-d5be832b38d4, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, edd03802-9025-4010-aa4d-282a26a72068)

root
 |-- trip_id: string (nullable = true)
 |-- load_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- truck_id: string (nullable = true)
 |-- trailer_id: string (nullable = true)
 |-- dispatch_date: date (nullable = true)
 |-- actual_distance_miles: integer (nullable = true)
 |-- actual_duration_hours: double (nullable = true)
 |-- fuel_gallons_used: double (nullable = true)
 |-- average_mpg: double (nullable = true)
 |-- idle_time_hours: double (nullable = true)
 |-- trip_status: string (nullable = true)

Source records: 85410


In [4]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    DateType
)

trips_path = (
    "abfss://Fleet_Logistics_Engineering@onelake.dfs.fabric.microsoft.com/"
    "Fleet_Logistics_Lakehouse.Lakehouse/Files/Landing/trips.csv"
)

trip_schema = StructType([
    StructField("trip_id", StringType(), True),
    StructField("load_id", StringType(), True),
    StructField("driver_id", StringType(), True),
    StructField("truck_id", StringType(), True),
    StructField("trailer_id", StringType(), True),
    StructField("dispatch_date", DateType(), True),
    StructField("actual_distance_miles", IntegerType(), True),
    StructField("actual_duration_hours", DoubleType(), True),
    StructField("fuel_gallons_used", DoubleType(), True),
    StructField("average_mpg", DoubleType(), True),
    StructField("idle_time_hours", DoubleType(), True),
    StructField("trip_status", StringType(), True)
])

StatementMeta(, 83a6f1df-6aca-48d1-879a-d5be832b38d4, 6, Finished, Available, Finished, False)

In [5]:
df_trips = (
    spark.read
    .option("header", "true")
    .schema(trip_schema)
    .csv(trips_path)
)

display(df_trips)

df_trips.printSchema()

print(f"Source records: {df_trips.count()}")

StatementMeta(, 83a6f1df-6aca-48d1-879a-d5be832b38d4, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2f051a04-b5c0-4c93-8da8-497d39bc3b10)

root
 |-- trip_id: string (nullable = true)
 |-- load_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- truck_id: string (nullable = true)
 |-- trailer_id: string (nullable = true)
 |-- dispatch_date: date (nullable = true)
 |-- actual_distance_miles: integer (nullable = true)
 |-- actual_duration_hours: double (nullable = true)
 |-- fuel_gallons_used: double (nullable = true)
 |-- average_mpg: double (nullable = true)
 |-- idle_time_hours: double (nullable = true)
 |-- trip_status: string (nullable = true)

Source records: 85410


In [6]:
null_trip_ids = (
    df_trips
    .filter(F.col("trip_id").isNull())
    .count()
)
print(f"NULL trip IDs: {null_trip_ids}")

StatementMeta(, 83a6f1df-6aca-48d1-879a-d5be832b38d4, 8, Finished, Available, Finished, False)

NULL trip IDs: 0


In [7]:
duplicate_trip_ids = (
    df_trips
    .groupBy("trip_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
print(f"Duplicate trip IDs: {duplicate_trip_ids}")

StatementMeta(, 83a6f1df-6aca-48d1-879a-d5be832b38d4, 9, Finished, Available, Finished, False)

Duplicate trip IDs: 0


In [8]:
missing_loads = (
    df_trips
    .join(
        spark.table("bronze_loads").select("load_id"),
        on="load_id",
        how="left_anti"
    )
    .count()
)

print(f"Trips with missing load IDs: {missing_loads}")

StatementMeta(, 83a6f1df-6aca-48d1-879a-d5be832b38d4, 10, Finished, Available, Finished, False)

Trips with missing load IDs: 0


In [9]:
missing_drivers = (
    df_trips
    .join(
        spark.table("bronze_drivers").select("driver_id"),
        on="driver_id",
        how="left_anti"
    )
    .count()
)

print(f"Trips with missing driver IDs: {missing_drivers}")

StatementMeta(, 83a6f1df-6aca-48d1-879a-d5be832b38d4, 11, Finished, Available, Finished, False)

Trips with missing driver IDs: 1714


In [10]:
missing_trucks = (
    df_trips
    .join(
        spark.table("bronze_trucks").select("truck_id"),
        on="truck_id",
        how="left_anti"
    )
    .count())

print(f"Trips with missing truck IDs: {missing_trucks}")

StatementMeta(, 83a6f1df-6aca-48d1-879a-d5be832b38d4, 12, Finished, Available, Finished, False)

Trips with missing truck IDs: 1672


In [11]:
missing_trailers = (
    df_trips
    .join(
        spark.table("bronze_trailers").select("trailer_id"),
        on="trailer_id",
        how="left_anti"
    )
    .count())

print(f"Trips with missing trailer IDs: {missing_trailers}")

StatementMeta(, 83a6f1df-6aca-48d1-879a-d5be832b38d4, 13, Finished, Available, Finished, False)

Trips with missing trailer IDs: 1680


In [12]:
invalid_distance = (
    df_trips
    .filter(F.col("actual_distance_miles") < 0)
    .count()
)
print(f"Invalid distance records: {invalid_distance}")

StatementMeta(, 83a6f1df-6aca-48d1-879a-d5be832b38d4, 14, Finished, Available, Finished, False)

Invalid distance records: 0


In [13]:
invalid_duration = (
    df_trips
    .filter(F.col("actual_duration_hours") < 0)
    .count()
)
print(f"Invalid duration records: {invalid_duration}")

StatementMeta(, 83a6f1df-6aca-48d1-879a-d5be832b38d4, 15, Finished, Available, Finished, False)

Invalid duration records: 0


In [14]:
invalid_fuel = (
    df_trips
    .filter(F.col("fuel_gallons_used") < 0)
    .count()
)
print(f"Invalid fuel records: {invalid_fuel}")

StatementMeta(, 83a6f1df-6aca-48d1-879a-d5be832b38d4, 16, Finished, Available, Finished, False)

Invalid fuel records: 0


In [15]:
invalid_mpg = (
    df_trips
    .filter(F.col("average_mpg") <= 0)
    .count()
)
print(f"Invalid MPG records: {invalid_mpg}")

StatementMeta(, 83a6f1df-6aca-48d1-879a-d5be832b38d4, 17, Finished, Available, Finished, False)

Invalid MPG records: 0


In [16]:
invalid_idle_time = (
    df_trips
    .filter(F.col("idle_time_hours") < 0)
    .count()
)
print(f"Invalid idle-time records: {invalid_idle_time}")

StatementMeta(, 83a6f1df-6aca-48d1-879a-d5be832b38d4, 18, Finished, Available, Finished, False)

Invalid idle-time records: 0


In [17]:
null_dispatch_dates = (
    df_trips
    .filter(F.col("dispatch_date").isNull())
    .count()
)
print(f"NULL dispatch dates: {null_dispatch_dates}")

StatementMeta(, 83a6f1df-6aca-48d1-879a-d5be832b38d4, 19, Finished, Available, Finished, False)

NULL dispatch dates: 0


In [18]:
if null_trip_ids > 0:
    raise ValueError("ETL failed: NULL trip_id values detected.")

if duplicate_trip_ids > 0:
    raise ValueError("ETL failed: Duplicate trip_id values detected.")

if missing_loads > 0:
    raise ValueError(
        "ETL failed: Trips contain load IDs not found in bronze_loads."
    )

if missing_drivers > 0:
    raise ValueError(
        "ETL failed: Trips contain driver IDs not found in bronze_drivers."
    )

if missing_trucks > 0:
    raise ValueError(
        "ETL failed: Trips contain truck IDs not found in bronze_trucks."
    )

if missing_trailers > 0:
    raise ValueError(
        "ETL failed: Trips contain trailer IDs not found in bronze_trailers."
    )

if invalid_distance > 0:
    raise ValueError("ETL failed: Negative distance values detected.")

if invalid_duration > 0:
    raise ValueError("ETL failed: Negative duration values detected.")

if invalid_fuel > 0:
    raise ValueError("ETL failed: Negative fuel consumption detected.")

if invalid_mpg > 0:
    raise ValueError("ETL failed: Invalid MPG values detected.")

if invalid_idle_time > 0:
    raise ValueError("ETL failed: Negative idle time detected.")

if null_dispatch_dates > 0:
    raise ValueError("ETL failed: NULL dispatch dates detected.")

print("Trip data quality and referential-integrity validation passed.")

StatementMeta(, 83a6f1df-6aca-48d1-879a-d5be832b38d4, 20, Finished, Available, Finished, False)

ValueError: ETL failed: Trips contain driver IDs not found in bronze_drivers.

In [19]:
missing_driver_details = (
    df_trips
    .join(
        spark.table("bronze_drivers").select("driver_id"),
        on="driver_id",
        how="left_anti"
    )
)

display(
    missing_driver_details
    .select("trip_id", "driver_id")
    .limit(20)
)

StatementMeta(, 83a6f1df-6aca-48d1-879a-d5be832b38d4, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 339d0c62-8050-44bb-8ada-1ac139f54513)

In [20]:
print(
    "Distinct missing driver IDs:",
    missing_driver_details
        .select("driver_id")
        .distinct()
        .count()
)

StatementMeta(, 83a6f1df-6aca-48d1-879a-d5be832b38d4, 22, Finished, Available, Finished, False)

Distinct missing driver IDs: 1


In [21]:
missing_truck_details = (
    df_trips
    .join(
        spark.table("bronze_trucks").select("truck_id"),
        on="truck_id",
        how="left_anti"
    )
)

display(
    missing_truck_details
    .select("trip_id", "truck_id")
    .limit(20)
)

print(
    "Distinct missing truck IDs:",
    missing_truck_details
        .select("truck_id")
        .distinct()
        .count()
)

StatementMeta(, 83a6f1df-6aca-48d1-879a-d5be832b38d4, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4660f905-0495-4d2a-8149-4fb65eecfb89)

Distinct missing truck IDs: 1


In [22]:
missing_trailer_details = (
    df_trips
    .join(
        spark.table("bronze_trailers").select("trailer_id"),
        on="trailer_id",
        how="left_anti"
    )
)

display(
    missing_trailer_details
    .select("trip_id", "trailer_id")
    .limit(20)
)

print(
    "Distinct missing trailer IDs:",
    missing_trailer_details
        .select("trailer_id")
        .distinct()
        .count()
)

StatementMeta(, 83a6f1df-6aca-48d1-879a-d5be832b38d4, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b595d27c-3d53-454f-ae17-6f78cfbb4af1)

Distinct missing trailer IDs: 1


In [23]:
# ============================================================
# CHECK NULL FOREIGN KEYS
# ============================================================

null_driver_ids = (
    df_trips
    .filter(F.col("driver_id").isNull())
    .count()
)

null_truck_ids = (
    df_trips
    .filter(F.col("truck_id").isNull())
    .count()
)

null_trailer_ids = (
    df_trips
    .filter(F.col("trailer_id").isNull())
    .count()
)

null_load_ids = (
    df_trips
    .filter(F.col("load_id").isNull())
    .count()
)

print(f"NULL driver IDs: {null_driver_ids}")
print(f"NULL truck IDs: {null_truck_ids}")
print(f"NULL trailer IDs: {null_trailer_ids}")
print(f"NULL load IDs: {null_load_ids}")

StatementMeta(, 83a6f1df-6aca-48d1-879a-d5be832b38d4, 25, Finished, Available, Finished, False)

NULL driver IDs: 1714
NULL truck IDs: 1672
NULL trailer IDs: 1680
NULL load IDs: 0


In [24]:
invalid_driver_ids = (
    df_trips
    .filter(F.col("driver_id").isNotNull())
    .join(
        spark.table("bronze_drivers").select("driver_id"),
        on="driver_id",
        how="left_anti"
    )
    .count()
)

print(f"Invalid non-NULL driver IDs: {invalid_driver_ids}")

StatementMeta(, 83a6f1df-6aca-48d1-879a-d5be832b38d4, 26, Finished, Available, Finished, False)

Invalid non-NULL driver IDs: 0


In [25]:
invalid_truck_ids = (
    df_trips
    .filter(F.col("truck_id").isNotNull())
    .join(
        spark.table("bronze_trucks").select("truck_id"),
        on="truck_id",
        how="left_anti"
    )
    .count()
)

print(f"Invalid non-NULL truck IDs: {invalid_truck_ids}")

StatementMeta(, 83a6f1df-6aca-48d1-879a-d5be832b38d4, 27, Finished, Available, Finished, False)

Invalid non-NULL truck IDs: 0


In [26]:
invalid_trailer_ids = (
    df_trips
    .filter(F.col("trailer_id").isNotNull())
    .join(
        spark.table("bronze_trailers").select("trailer_id"),
        on="trailer_id",
        how="left_anti"
    )
    .count()
)

print(f"Invalid non-NULL trailer IDs: {invalid_trailer_ids}")

StatementMeta(, 83a6f1df-6aca-48d1-879a-d5be832b38d4, 28, Finished, Available, Finished, False)

Invalid non-NULL trailer IDs: 0


In [27]:
invalid_load_ids = (
    df_trips
    .filter(F.col("load_id").isNotNull())
    .join(
        spark.table("bronze_loads").select("load_id"),
        on="load_id",
        how="left_anti"
    )
    .count()
)

print(f"Invalid non-NULL load IDs: {invalid_load_ids}")

StatementMeta(, 83a6f1df-6aca-48d1-879a-d5be832b38d4, 29, Finished, Available, Finished, False)

Invalid non-NULL load IDs: 0


In [28]:
# ============================================================
# FINAL TRIP ETL VALIDATION
# ============================================================

if null_trip_ids > 0:
    raise ValueError("ETL failed: NULL trip_id values detected.")

if duplicate_trip_ids > 0:
    raise ValueError("ETL failed: Duplicate trip_id values detected.")

if invalid_load_ids > 0:
    raise ValueError(
        "ETL failed: Invalid non-NULL load IDs detected."
    )

if invalid_driver_ids > 0:
    raise ValueError(
        "ETL failed: Invalid non-NULL driver IDs detected."
    )

if invalid_truck_ids > 0:
    raise ValueError(
        "ETL failed: Invalid non-NULL truck IDs detected."
    )

if invalid_trailer_ids > 0:
    raise ValueError(
        "ETL failed: Invalid non-NULL trailer IDs detected."
    )

if invalid_distance > 0:
    raise ValueError("ETL failed: Negative distance values detected.")

if invalid_duration > 0:
    raise ValueError("ETL failed: Negative duration values detected.")

if invalid_fuel > 0:
    raise ValueError("ETL failed: Negative fuel consumption detected.")

if invalid_mpg > 0:
    raise ValueError("ETL failed: Invalid MPG values detected.")

if invalid_idle_time > 0:
    raise ValueError("ETL failed: Negative idle time detected.")

if null_dispatch_dates > 0:
    raise ValueError("ETL failed: NULL dispatch dates detected.")

print("Trip data quality and referential-integrity validation passed.")

StatementMeta(, 83a6f1df-6aca-48d1-879a-d5be832b38d4, 30, Finished, Available, Finished, False)

Trip data quality and referential-integrity validation passed.


In [29]:
from delta.tables import DeltaTable

# ------------------------------------------------------------
# Add ETL metadata
# ------------------------------------------------------------

df_trips_bronze = (
    df_trips
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file", F.lit("trips.csv"))
)

# Create temporary source view
df_trips_bronze.createOrReplaceTempView("trips_source")

# ------------------------------------------------------------
# Create Bronze table if it does not already exist
# ------------------------------------------------------------

spark.sql("""
CREATE TABLE IF NOT EXISTS bronze_trips (
    trip_id STRING,
    load_id STRING,
    driver_id STRING,
    truck_id STRING,
    trailer_id STRING,
    dispatch_date DATE,
    actual_distance_miles INT,
    actual_duration_hours DOUBLE,
    fuel_gallons_used DOUBLE,
    average_mpg DOUBLE,
    idle_time_hours DOUBLE,
    trip_status STRING,
    ingestion_timestamp TIMESTAMP,
    source_file STRING
)
""")


StatementMeta(, 83a6f1df-6aca-48d1-879a-d5be832b38d4, 31, Finished, Available, Finished, False)

DataFrame[]

In [30]:
spark.sql("""
MERGE INTO bronze_trips AS target

USING trips_source AS source

ON target.trip_id = source.trip_id

WHEN MATCHED THEN
    UPDATE SET
        target.load_id = source.load_id,
        target.driver_id = source.driver_id,
        target.truck_id = source.truck_id,
        target.trailer_id = source.trailer_id,
        target.dispatch_date = source.dispatch_date,
        target.actual_distance_miles = source.actual_distance_miles,
        target.actual_duration_hours = source.actual_duration_hours,
        target.fuel_gallons_used = source.fuel_gallons_used,
        target.average_mpg = source.average_mpg,
        target.idle_time_hours = source.idle_time_hours,
        target.trip_status = source.trip_status,
        target.ingestion_timestamp = source.ingestion_timestamp,
        target.source_file = source.source_file

WHEN NOT MATCHED THEN
    INSERT (
        trip_id,
        load_id,
        driver_id,
        truck_id,
        trailer_id,
        dispatch_date,
        actual_distance_miles,
        actual_duration_hours,
        fuel_gallons_used,
        average_mpg,
        idle_time_hours,
        trip_status,
        ingestion_timestamp,
        source_file
    )

    VALUES (
        source.trip_id,
        source.load_id,
        source.driver_id,
        source.truck_id,
        source.trailer_id,
        source.dispatch_date,
        source.actual_distance_miles,
        source.actual_duration_hours,
        source.fuel_gallons_used,
        source.average_mpg,
        source.idle_time_hours,
        source.trip_status,
        source.ingestion_timestamp,
        source.source_file
    )
""")

print("Trip Bronze MERGE completed successfully.")

# ------------------------------------------------------------
# Verify Bronze record count
# ------------------------------------------------------------

bronze_trip_count = spark.table("bronze_trips").count()

print(f"Bronze trip records: {bronze_trip_count}")

StatementMeta(, 83a6f1df-6aca-48d1-879a-d5be832b38d4, 32, Finished, Available, Finished, False)

Trip Bronze MERGE completed successfully.
Bronze trip records: 85410
